# 面试问题：倒排索引如何支持字段检索、短语匹配与增量更新？

可以直接复述的回答是：正排表按文档保存字段，倒排表按 term 保存 docID、词频与位置。查询时先读取各 term postings，再做交集或并集形成候选，避免扫描全部文档。字段权重和位置能区分标题命中、正文命中与短语连续命中。库存、ACL 等动态约束应在候选阶段过滤，但索引版本必须和正排记录一致。增量更新不能只追加新 postings，还要移除旧版本词项或写 tombstone，否则已删除内容会继续被召回。下面用八件商品手写 positional inverted index，并真实复现 stale posting。

## 真实案例：电商商品关键词搜索与标题改版

八条脱敏商品记录包含标题、正文、库存和版本。文本预先用空格标出教学分词边界，避免把中文分词器本身混入索引主题；生产中应替换为版本化 tokenizer。

In [1]:
from collections import defaultdict  # 导入倒排桶和位置列表需要的默认字典
documents = {  # 定义八条带真实字段的商品正排记录
    "P-101": {"title": "无线 降噪 耳机", "body": "通勤 蓝牙 主动 降噪", "stock": 12, "version": 1},  # 无线耳机主商品
    "P-102": {"title": "头戴 降噪 耳机", "body": "录音室 有线 舒适 耳罩", "stock": 7, "version": 1},  # 头戴有线耳机
    "P-103": {"title": "机械 键盘", "body": "办公 蓝牙 茶轴 键盘", "stock": 18, "version": 1},  # 办公键盘商品
    "P-104": {"title": "便携 咖啡机", "body": "露营 手压 浓缩 咖啡", "stock": 5, "version": 1},  # 户外咖啡机
    "P-105": {"title": "扫地 机器人", "body": "激光 导航 自动 集尘", "stock": 0, "version": 1},  # 缺货扫地机器人
    "P-106": {"title": "运动 相机", "body": "防水 防抖 骑行 录像", "stock": 9, "version": 1},  # 运动相机
    "P-107": {"title": "蓝牙 音箱", "body": "户外 防水 长续航 音箱", "stock": 14, "version": 1},  # 户外音箱
    "P-108": {"title": "无线 充电 宝", "body": "磁吸 快充 旅行 电源", "stock": 11, "version": 1},  # 无线充电商品
}  # 结束八条商品记录
queries = [  # 定义六个具有明确相关商品的搜索请求
    ("Q-01", "无线 降噪 耳机", "P-101"),  # 三词标题短语查询
    ("Q-02", "蓝牙 键盘", "P-103"),  # 标题和正文混合查询
    ("Q-03", "便携 咖啡机", "P-104"),  # 两词标题查询
    ("Q-04", "防水 相机", "P-106"),  # 正文与标题跨字段查询
    ("Q-05", "户外 音箱", "P-107"),  # 正文和标题联合查询
    ("Q-06", "无线 充电", "P-108"),  # 相邻标题词查询
]  # 结束六个查询
def tokenize(text):  # 定义与索引版本绑定的教学 tokenizer
    return [token for token in text.lower().split() if token]  # 按显式空格得到稳定词项
print("输入预览：doc | version | stock | title | body")  # 输出商品记录表头
for doc_id, document in documents.items():  # 逐条展示八件商品
    print(f"{doc_id} | v{document['version']} | {document['stock']:2d} | {document['title']} | {document['body']}")  # 展示可读正排字段
print("查询：", [(query_id, text, expected) for query_id, text, expected in queries])  # 展示六个搜索目标

输入预览：doc | version | stock | title | body
P-101 | v1 | 12 | 无线 降噪 耳机 | 通勤 蓝牙 主动 降噪
P-102 | v1 |  7 | 头戴 降噪 耳机 | 录音室 有线 舒适 耳罩
P-103 | v1 | 18 | 机械 键盘 | 办公 蓝牙 茶轴 键盘
P-104 | v1 |  5 | 便携 咖啡机 | 露营 手压 浓缩 咖啡
P-105 | v1 |  0 | 扫地 机器人 | 激光 导航 自动 集尘
P-106 | v1 |  9 | 运动 相机 | 防水 防抖 骑行 录像
P-107 | v1 | 14 | 蓝牙 音箱 | 户外 防水 长续航 音箱
P-108 | v1 | 11 | 无线 充电 宝 | 磁吸 快充 旅行 电源
查询： [('Q-01', '无线 降噪 耳机', 'P-101'), ('Q-02', '蓝牙 键盘', 'P-103'), ('Q-03', '便携 咖啡机', 'P-104'), ('Q-04', '防水 相机', 'P-106'), ('Q-05', '户外 音箱', 'P-107'), ('Q-06', '无线 充电', 'P-108')]


## Baseline / 基线：逐文档扫描全部词项

基线对每个查询读取八条正排记录并检查所有 query term，结果正确但查询成本随文档总量线性增长。

In [2]:
def scan_search(query, source_documents):  # 实现无索引的全文扫描基线
    query_terms = tokenize(query)  # 对查询执行同版本分词
    matches = []  # 收集满足全部词项且有库存的商品
    scanned = 0  # 统计实际读取的正排记录数
    for doc_id, document in source_documents.items():  # 逐条扫描全部商品
        scanned += 1  # 每检查一条正排记录增加一次扫描
        all_terms = tokenize(document["title"] + " " + document["body"])  # 合并标题和正文词项
        if document["stock"] > 0 and all(term in all_terms for term in query_terms):  # 同时应用 AND 匹配和库存过滤
            matches.append(doc_id)  # 保存命中商品身份
    return matches, scanned  # 返回命中列表和扫描成本
baseline_rows = []  # 保存六个查询基线结果
for query_id, text, expected in queries:  # 遍历同一组搜索请求
    matches, scanned = scan_search(text, documents)  # 对当前查询执行全文扫描
    baseline_rows.append((query_id, matches, scanned, expected in matches))  # 保存候选、成本和目标召回
print("query | matches | scanned_docs | expected_hit")  # 输出全文扫描结果表头
for row in baseline_rows:  # 逐查询展示基线行为
    print(f"{row[0]} | {row[1]} | {row[2]} | {row[3]}")  # 展示每次都扫描八条记录

query | matches | scanned_docs | expected_hit
Q-01 | ['P-101'] | 8 | True
Q-02 | ['P-103'] | 8 | True
Q-03 | ['P-104'] | 8 | True
Q-04 | ['P-106'] | 8 | True
Q-05 | ['P-107'] | 8 | True
Q-06 | ['P-108'] | 8 | True


## 核心实现：字段级 positional postings 与候选交集

postings 结构为 `term -> field -> docID -> positions`。标题词频乘 2，正文词频乘 1；查询词在标题连续出现时再加 3 分。输出 Q-01 每个词项的 postings 与位置。

In [3]:
def add_document(index, doc_id, document):  # 把一条正排记录写入字段位置倒排表
    for field in ["title", "body"]:  # 分别处理标题和正文字段
        for position, term in enumerate(tokenize(document[field])):  # 遍历字段中的词项与位置
            index[term][field][doc_id].append(position)  # 追加当前 docID 的词项位置
def build_index(source_documents):  # 从正排表构建完整倒排索引
    index = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))  # 创建 term-field-doc-position 多级桶
    for doc_id, document in source_documents.items():  # 遍历全部初始商品
        add_document(index, doc_id, document)  # 写入当前商品的字段 postings
    return index  # 返回可增量维护的倒排表
inverted_index = build_index(documents)  # 对八条商品建立初始索引
def has_title_phrase(doc_id, query_terms, index):  # 判断查询词是否在标题中连续出现
    position_lists = [index[term]["title"].get(doc_id, []) for term in query_terms]  # 读取每个查询词的标题位置
    if any(not positions for positions in position_lists):  # 任一词不在标题时无法构成短语
        return False  # 返回标题短语不命中
    return any(all(start + offset in position_lists[offset] for offset in range(len(query_terms))) for start in position_lists[0])  # 检查连续位置链
def indexed_search(query, source_documents, index):  # 使用倒排交集执行字段加权搜索
    query_terms = tokenize(query)  # 获取查询词项
    posting_sets = []  # 收集每个词项出现的文档集合
    posting_reads = 0  # 统计读取的 posting 条目数量
    for term in query_terms:  # 依次读取每个查询词 postings
        term_docs = set(index[term]["title"]) | set(index[term]["body"])  # 合并标题与正文 docID
        posting_sets.append(term_docs)  # 保存当前词项候选集合
        posting_reads += len(term_docs)  # 累加真实 posting 访问量
    candidates = set.intersection(*posting_sets) if posting_sets else set()  # 对所有查询词执行 AND 交集
    scored = []  # 收集库存过滤后的字段分数
    for doc_id in candidates:  # 只遍历倒排交集候选而非全部商品
        if source_documents[doc_id]["stock"] <= 0:  # 检查动态库存门禁
            continue  # 缺货商品不进入最终排名
        score = 0.0  # 初始化当前商品相关性分数
        for term in query_terms:  # 累加每个词项的字段词频
            score += 2.0 * len(index[term]["title"].get(doc_id, []))  # 标题命中获得双倍权重
            score += 1.0 * len(index[term]["body"].get(doc_id, []))  # 正文命中获得基础权重
        if has_title_phrase(doc_id, query_terms, index):  # 检查标题位置是否连续
            score += 3.0  # 对完整标题短语增加奖励
        scored.append((doc_id, score))  # 保存候选身份与可解释分数
    scored.sort(key=lambda item: (-item[1], item[0]))  # 按分数降序并用 docID 稳定打破平局
    return scored, candidates, posting_reads  # 返回排名、原始候选和 posting 成本
print("Q-01 postings：term | title(doc:positions) | body(doc:positions)")  # 输出位置倒排中间量表头
for term in tokenize(queries[0][1]):  # 遍历首个查询的三个词
    title_postings = {doc_id: positions for doc_id, positions in inverted_index[term]["title"].items()}  # 提取标题位置表
    body_postings = {doc_id: positions for doc_id, positions in inverted_index[term]["body"].items()}  # 提取正文位置表
    print(f"{term} | {title_postings} | {body_postings}")  # 展示真实 postings 和词位
q1_ranked, q1_candidates, q1_reads = indexed_search(queries[0][1], documents, inverted_index)  # 执行首个索引查询
print("Q-01 候选交集：", sorted(q1_candidates), "排名：", q1_ranked, "posting_reads：", q1_reads)  # 展示候选形成与字段分数

Q-01 postings：term | title(doc:positions) | body(doc:positions)
无线 | {'P-101': [0], 'P-108': [0]} | {}
降噪 | {'P-101': [1], 'P-102': [1]} | {'P-101': [3]}
耳机 | {'P-101': [2], 'P-102': [2]} | {}
Q-01 候选交集： ['P-101'] 排名： [('P-101', 10.0)] posting_reads： 6


## 结果表：全文扫描与倒排检索使用同一数据

In [4]:
indexed_rows = []  # 收集六个查询的倒排结果
print("query | baseline_top | index_top | postings_read | full_scan | phrase_bonus")  # 输出两种方案统一对照表头
for query_id, text, expected in queries:  # 遍历六个搜索请求
    baseline_matches, scanned = scan_search(text, documents)  # 获取同数据全文扫描结果
    ranked, candidates, posting_reads = indexed_search(text, documents, inverted_index)  # 获取倒排候选和排名
    top_id = ranked[0][0] if ranked else None  # 提取倒排第一名商品
    phrase_bonus = bool(top_id and has_title_phrase(top_id, tokenize(text), inverted_index))  # 检查第一名是否获得标题短语奖励
    indexed_rows.append((query_id, top_id, expected, posting_reads, scanned))  # 保存逐查询评估记录
    baseline_top = baseline_matches[0] if baseline_matches else None  # 提取扫描基线第一命中
    print(f"{query_id} | {baseline_top} | {top_id} | {posting_reads:13d} | {scanned:9d} | {phrase_bonus}")  # 展示候选成本和目标结果
index_accuracy = sum(top_id == expected for query_id, top_id, expected, posting_reads, scanned in indexed_rows) / len(indexed_rows)  # 计算六查询首位准确率
average_posting_reads = sum(row[3] for row in indexed_rows) / len(indexed_rows)  # 计算平均 posting 访问量
print(f"教学实验 top1={index_accuracy:.1%}，平均 postings={average_posting_reads:.1f}，全文扫描={len(documents)}")  # 汇总结果与成本

query | baseline_top | index_top | postings_read | full_scan | phrase_bonus
Q-01 | P-101 | P-101 |             6 |         8 | True
Q-02 | P-103 | P-103 |             4 |         8 | False
Q-03 | P-104 | P-104 |             2 |         8 | True
Q-04 | P-106 | P-106 |             3 |         8 | False
Q-05 | P-107 | P-107 |             2 |         8 | False
Q-06 | P-108 | P-108 |             3 |         8 | True
教学实验 top1=100.0%，平均 postings=3.3，全文扫描=8


## 失败案例与修正：商品改版后旧 postings 未删除

P-101 从“无线降噪耳机”改成“无线运动耳机”。错误增量逻辑只追加新版词项，因此旧的“降噪”posting 仍会和“无线”相交。修正逻辑先从所有 postings 移除该 docID，再写入新版本。

In [5]:
updated_p101 = {"title": "无线 运动 耳机", "body": "跑步 蓝牙 防汗", "stock": 12, "version": 2}  # 定义删除降噪卖点的新版商品
naive_documents = {doc_id: document.copy() for doc_id, document in documents.items()}  # 复制正排表用于失败实验
naive_documents["P-101"] = updated_p101  # 将正排记录更新到版本二
naive_index = build_index(documents)  # 从旧版本建立倒排表
add_document(naive_index, "P-101", updated_p101)  # 错误地只追加新版 postings
stale_ranked, stale_candidates, stale_reads = indexed_search("无线 降噪", naive_documents, naive_index)  # 搜索已经不存在的旧卖点组合
fixed_index = build_index(documents)  # 重建独立索引用于正确增量更新
def replace_document(index, doc_id, new_document):  # 手写安全替换一条商品索引的方法
    for term in list(index.keys()):  # 遍历当前全部词项桶
        for field in ["title", "body"]:  # 检查标题和正文 postings
            index[term][field].pop(doc_id, None)  # 删除该 docID 的所有旧位置记录
        if not index[term]["title"] and not index[term]["body"]:  # 检查词项是否已经没有任何文档
            index.pop(term)  # 清理空词项桶避免字典膨胀
    add_document(index, doc_id, new_document)  # 在清理旧版本后写入新版 postings
fixed_documents = {doc_id: document.copy() for doc_id, document in documents.items()}  # 复制正排表用于正确更新
fixed_documents["P-101"] = updated_p101  # 提交新版正排记录
replace_document(fixed_index, "P-101", updated_p101)  # 对倒排表执行先删后加
fixed_ranked, fixed_candidates, fixed_reads = indexed_search("无线 降噪", fixed_documents, fixed_index)  # 再次搜索已删除旧卖点
scan_after_update, scan_cost_after_update = scan_search("无线 降噪", fixed_documents)  # 用正排扫描确认权威结果
print("错误追加：候选=", sorted(stale_candidates), "排名=", stale_ranked)  # 展示 stale posting 仍召回 P-101
print("先删后加：候选=", sorted(fixed_candidates), "排名=", fixed_ranked)  # 展示修正后的倒排结果
print("更新后正排扫描权威结果：", scan_after_update)  # 对照正排事实验证修正

错误追加：候选= ['P-101'] 排名= [('P-101', 10.0)]
先删后加：候选= [] 排名= []
更新后正排扫描权威结果： []


## 结果解读

倒排查询只读取相关词项 postings，再做集合交集和字段打分；短语位置让“无线 充电”优先于分散命中。失败实验说明正排与倒排不是自动一致，更新协议本身就是搜索正确性的一部分。

## 生产边界

教学索引只有八条记录且全部驻留内存。生产系统还需分片、压缩 postings、skip list、BM25、tokenizer 版本、刷新间隔、WAL、段合并、删除位图、ACL 与库存缓存。更新必须有文档版本或序列号，监控 stale hit rate，并支持从正排快照重建与校验。

## 最小回归测试

In [6]:
assert len(documents) >= 6 and len(queries) >= 6  # 保证案例覆盖多个商品与查询
assert index_accuracy == 1.0  # 保证六个查询的倒排第一名符合人工目标
assert q1_candidates == {"P-101"} and q1_ranked[0][1] >= 9.0  # 保证词项交集和标题短语奖励真实生效
assert average_posting_reads < len(documents)  # 保证平均 posting 访问量小于全文扫描文档数
assert "P-101" in stale_candidates  # 保证只追加更新会复现旧词项污染
assert "P-101" not in fixed_candidates and fixed_ranked == []  # 保证先删后加移除 stale posting
assert fixed_ranked == [(doc_id, score) for doc_id, score in fixed_ranked if doc_id in scan_after_update]  # 保证更新后索引结果与正排事实一致